In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
file_path = "../data/raw/project_master_dataset.csv"
df = pd.read_csv(file_path)

print("Dataset Loaded.")
print("Shape:", df.shape)

df.head()

Dataset Loaded.
Shape: (100, 13)


,project_id,project_title,project_type,deadline_days,base_revenue,complexity_level,client_priority,team_experience_level,historical_delay_rate,actual_completion_days,completed_on_time,delay_days,final_revenue_realized
0,1,E-Commerce Platform,Web Development,3,80000,Medium,High,Senior Team,15%,2,Yes,0,80000
1,2,Payment API Integration,Backend Development,2,60000,High,Very High,Mid-Level Team,30%,3,No,1,54000
2,3,Mobile UI Enhancement,Mobile Development,1,50000,Low,Medium,Senior Team,10%,1,Yes,0,50000
3,4,CRM System Upgrade,Web Development,4,100000,Very High,Very High,Junior Team,40%,5,No,1,90000
4,5,SEO Optimization,Marketing,2,30000,Very Low,Low,Senior Team,5%,2,Yes,0,30000


In [3]:
# Convert percentage to numeric
df["historical_delay_rate"] = (
    df["historical_delay_rate"]
    .str.replace("%", "")
    .astype(float) / 100
)

# Keep original categorical labels for training
print("Basic cleaning completed.")
df.head()

Basic cleaning completed.


,project_id,project_title,project_type,deadline_days,base_revenue,complexity_level,client_priority,team_experience_level,historical_delay_rate,actual_completion_days,completed_on_time,delay_days,final_revenue_realized
0,1,E-Commerce Platform,Web Development,3,80000,Medium,High,Senior Team,0.15,2,Yes,0,80000
1,2,Payment API Integration,Backend Development,2,60000,High,Very High,Mid-Level Team,0.30,3,No,1,54000
2,3,Mobile UI Enhancement,Mobile Development,1,50000,Low,Medium,Senior Team,0.10,1,Yes,0,50000
3,4,CRM System Upgrade,Web Development,4,100000,Very High,Very High,Junior Team,0.40,5,No,1,90000
4,5,SEO Optimization,Marketing,2,30000,Very Low,Low,Senior Team,0.05,2,Yes,0,30000


## Encode Target Columns

In [4]:
target_encoders = {}

target_columns = [
    "complexity_level",
    "client_priority",
    "team_experience_level"
]

for col in target_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    target_encoders[col] = le

# Save encoders for API decoding later
joblib.dump(target_encoders, "../models/target_label_encoders.pkl")

print("Target columns encoded and encoders saved.")

Target columns encoded and encoders saved.


## Define Preprocessing Pipeline

In [5]:
# Feature columns
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
feature_columns = [
    "project_title",
    "project_type",
    "deadline_days",
    "base_revenue"
]

X = df[feature_columns]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "project_title"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["project_type"]),
        ("num", StandardScaler(), ["deadline_days", "base_revenue"])
    ]
)

print("Preprocessing pipeline defined.")

Preprocessing pipeline defined.


In [6]:
from sklearn.base import clone

y_complexity = df["complexity_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_complexity, test_size=0.2, random_state=42
)

complexity_pipeline = Pipeline(
    steps=[
        ("features", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

complexity_pipeline.fit(X_train, y_train)

print("Complexity model trained.")

Complexity model trained.


In [7]:
y_pred = complexity_pipeline.predict(X_test)

print("Complexity Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Complexity Accuracy: 0.95

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       0.88      1.00      0.93         7
           2       1.00      1.00      1.00         6
           3       1.00      1.00      1.00         1
           4       0.00      0.00      0.00         1

    accuracy                           0.95        20
   macro avg       0.78      0.80      0.79        20
weighted avg       0.91      0.95      0.93        20



C:\Users\dipes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\dipes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\dipes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [8]:
joblib.dump(complexity_pipeline, "../models/model_complexity.pkl")
print("Complexity model saved.")

Complexity model saved.


## MODEL 2 — Client Priority

In [9]:
from sklearn.base import clone

y_priority = df["client_priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42
)

priority_pipeline = Pipeline(
    steps=[
        ("features", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

priority_pipeline.fit(X_train, y_train)

print("Client Priority model trained.")

Client Priority model trained.


In [10]:
y_pred = priority_pipeline.predict(X_test)

print("Client Priority Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Client Priority Accuracy: 0.8

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.78      0.82         9
           1       0.00      0.00      0.00         1
           2       0.75      1.00      0.86         6
           3       0.75      0.75      0.75         4

    accuracy                           0.80        20
   macro avg       0.59      0.63      0.61        20
weighted avg       0.77      0.80      0.78        20



C:\Users\dipes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\dipes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\dipes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [11]:
joblib.dump(priority_pipeline, "../models/model_client_priority.pkl")
print("Client Priority model saved.")

Client Priority model saved.


## MODEL 3 — Team Experience Level

In [12]:
from sklearn.base import clone

y_team = df["team_experience_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_team,
    test_size=0.2,
    random_state=42,
    stratify=y_team
)

team_pipeline_v2 = Pipeline(
    steps=[
        ("features", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

team_pipeline_v2.fit(X_train, y_train)

print("Team Experience model trained.")

Team Experience model trained.


In [13]:
y_pred = team_pipeline_v2.predict(X_test)

print("Team Experience Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Team Experience Accuracy: 0.75

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.33      0.25      0.29         4
           2       0.81      0.93      0.87        14

    accuracy                           0.75        20
   macro avg       0.72      0.56      0.61        20
weighted avg       0.74      0.75      0.73        20



In [14]:
joblib.dump(team_pipeline_v2, "../models/model_team_experience.pkl")
print("Team Experience model saved.")

Team Experience model saved.


## FINAL TEST — Simulate New Project

In [15]:
# Example new project
new_project = pd.DataFrame([{
    "project_title": "AI Fraud Detection Engine",
    "project_type": "AI Development",
    "deadline_days": 4,
    "base_revenue": 150000
}])

# Predict
pred_complexity = complexity_pipeline.predict(new_project)
pred_priority = priority_pipeline.predict(new_project)
pred_team = team_pipeline_v2.predict(new_project)

# Decode back to original labels
encoders = joblib.load("../models/target_label_encoders.pkl")

decoded_complexity = encoders["complexity_level"].inverse_transform(pred_complexity)
decoded_priority = encoders["client_priority"].inverse_transform(pred_priority)
decoded_team = encoders["team_experience_level"].inverse_transform(pred_team)

print("Predicted Complexity:", decoded_complexity[0])
print("Predicted Client Priority:", decoded_priority[0])
print("Predicted Team Experience:", decoded_team[0])

Predicted Complexity: Very High
Predicted Client Priority: Very High
Predicted Team Experience: Senior Team


## 🟢 Model 4 — Predict historical_delay_rate (Regression)

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

feature_cols_delay = [
    "project_type",
    "complexity_level",
    "team_experience_level",
    "client_priority"
]

X_delay = df[feature_cols_delay]
y_delay = df["historical_delay_rate"]

X_train_delay, X_test_delay, y_train_delay, y_test_delay = train_test_split(
    X_delay,
    y_delay,
    test_size=0.2,
    random_state=42
)



In [19]:
# Encode Categorical Columns

from sklearn.preprocessing import OneHotEncoder

delay_encoder = OneHotEncoder(handle_unknown="ignore")

X_train_encoded = delay_encoder.fit_transform(X_train_delay)
X_test_encoded = delay_encoder.transform(X_test_delay)

In [20]:
# Train Regressor

delay_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

delay_model.fit(X_train_encoded, y_train_delay)

print("Delay rate model trained.")

Delay rate model trained.


In [21]:
y_pred_delay = delay_model.predict(X_test_encoded)

rmse_delay = np.sqrt(mean_squared_error(y_test_delay, y_pred_delay))

print("Delay Rate RMSE:", rmse_delay)

Delay Rate RMSE: 0.026525858994413655


In [22]:
joblib.dump(delay_model, "../models/model_delay_rate.pkl")
joblib.dump(delay_encoder, "../models/delay_encoder.pkl")

print("Delay model saved.")

Delay model saved.


## MODEL 5 — Predict actual_completion_days

In [25]:
feature_cols_completion = [
    "deadline_days",
    "complexity_level",
    "team_experience_level",
    "historical_delay_rate"
]

X_comp = df[feature_cols_completion]
y_comp = df["actual_completion_days"]

X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
    X_comp,
    y_comp,
    test_size=0.2,
    random_state=42
)

In [26]:
completion_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

completion_model.fit(X_train_comp, y_train_comp)

print("Completion days model trained.")

Completion days model trained.


In [27]:
y_pred_comp = completion_model.predict(X_test_comp)

rmse_comp = np.sqrt(mean_squared_error(y_test_comp, y_pred_comp))

print("Completion Days RMSE:", rmse_comp)

Completion Days RMSE: 0.41999503624509854


In [28]:
joblib.dump(completion_model, "../models/model_completion_days.pkl")

print("Completion model saved.")

Completion model saved.
